# Day 4 - Hash sets, hash maps, and how LLMs are made

**Big idea (coding):** when a problem asks "have I seen this before?", a set or dict turns a slow nested loop into one fast pass.

## 1. Contains Duplicate

*Does any number appear twice?*

- **Slow way:** compare every pair -> O(n^2). 10,000 numbers = ~50 million comparisons.
- **Fast way:** walk once and remember what you've seen in a **set**. "Is it in the set?" is instant (O(1)), so the whole thing is O(n).

In [1]:
def contains_duplicate_slow(nums):
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] == nums[j]:
                return True
    return False

def contains_duplicate(nums):
    seen = set()
    for n in nums:
        if n in seen:
            return True
        seen.add(n)
    return False

for case in [[1, 2, 3, 1], [1, 2, 3, 4], []]:
    print(case, "->", contains_duplicate(case))

[1, 2, 3, 1] -> True
[1, 2, 3, 4] -> False
[] -> False


Race them on 5,000 numbers with no duplicates (the worst case - both have to check everything):

In [2]:
import time

nums = list(range(5_000))

for fn in (contains_duplicate_slow, contains_duplicate):
    start = time.perf_counter()
    fn(nums)
    print(f"{fn.__name__:<26} {time.perf_counter() - start:.4f} s")

contains_duplicate_slow    0.3562 s
contains_duplicate         0.0002 s


## 2. Two Sum

*Find two numbers that add up to `target` and return their positions.*

The trick: for each number `n`, the partner you need is `target - n` (the **complement**). Keep a **dict** of `value -> index` for everything seen so far. If the complement is already in the dict, you're done.

In [3]:
def two_sum(nums, target, show=False):
    seen = {}   # value -> index
    for i, n in enumerate(nums):
        complement = target - n
        if show:
            print(f"  i={i} n={n:<3} need {complement:<3} seen={seen}")
        if complement in seen:
            return [seen[complement], i]
        seen[n] = i
    return []

print("answer:", two_sum([2, 7, 11, 15], 9, show=True))
print("answer:", two_sum([3, 2, 4], 6, show=True))

  i=0 n=2   need 7   seen={}
  i=1 n=7   need 2   seen={2: 0}
answer: [0, 1]
  i=0 n=3   need 3   seen={}
  i=1 n=2   need 4   seen={3: 0}
  i=2 n=4   need 2   seen={3: 0, 2: 1}
answer: [1, 2]


**Pattern to remember:** use a **set** when you only need to know *whether* you've seen something. Use a **dict** when you also need to know *where*.

## 3. Karpathy - "Intro to Large Language Models" (the simple version)

1. **An LLM is two files:** a huge file of numbers (the weights) + a small program that runs them.
2. **Training is expensive, using it is cheap:** thousands of GPUs for weeks to train; one machine to run.
3. **It's compression:** ~10 TB of internet text squeezed into the weights. Patterns survive, exact facts get blurry.
4. **So it hallucinates:** it predicts *plausible* next words; it doesn't look facts up.
5. **Two stages:** pretraining makes a "document completer"; fine-tuning on Q&A conversations (plus RLHF) turns it into an assistant.
6. **Scaling laws:** more parameters + more data -> predictably better. That's why everyone wants more GPUs.
7. **Tools:** models now call search, calculators, and code instead of doing everything from memory.
8. **System 1 vs System 2:** today's models answer on reflex; the goal is models that "think longer" for better answers.
9. **LLM as an operating system:** the model is the kernel, the context window is RAM, tools are the apps.
10. **Security:** jailbreaks, prompt injection (hidden instructions in a web page), data poisoning - still unsolved.

## Recap

- Seen before? -> `set`. Seen before, and where? -> `dict`. O(n^2) becomes O(n).
- An LLM = weights + runner. Pretrain -> fine-tune. It predicts; it doesn't look things up.